## Importing Functions

In [ ]:
from __future__ import annotations

import warnings
warnings.filterwarnings(
    "ignore",
    message=r"^`sklearn\.utils\.parallel\.delayed` should be used with `sklearn\.utils\.parallel\.Parallel`.*",
    category=UserWarning,
)

from pathlib import Path
from typing import Sequence, Union
import json
import warnings
import joblib
import pandas as pd
import rasterio
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score

from functions.gpkg_funcs import (
    import_gpkg_func,
)

from functions.s2_import_funcs import (
    sentinel_path_builder,
)

from functions.ML_classification_check_funcs import (
    ml_classification_check,
)

from notebooks_dir._04_ML_classification._support._n01_funcs import (
    expected_crs_from_input,
    RF_models,
    sample_training_bands_for_RF,
    apply_RF_model_to_rstrs,
)

###############################################################################
## FULL WORKFLOW
###############################################################################
def RF_trainer_on_single_year(
    *,
    # Raster stack inputs
    stacked_raster_dir: Union[str, Path],
    stacked_raster_crs: str,
    stacked_raster_filename: str,
    all_raster_band_names: Sequence[str],
    feature_band_names: Sequence[str],
    years: Sequence[Union[int, str]],
    timeframes: Sequence[Union[str, int]],
    # Valid mask
    valid_band_available: bool = True,
    valid_band_name: str = "VALID_MASK",
    valid_value: int = 1,
    # Train/val vectors
    train_gpkg_path: Union[str, Path],
    val_gpkg_path: Union[str, Path],
    gpkg_label_col: str = "type",
    # RF setup
    rf_model_selection: str = "mdl1",
    train_year: int = 2017,
    hab_selection: str = "WD1",
    # Naming controls
    stack_description: str = "b2348",
    train_split_attempt: str = "atX",
    # Output dirs (parent folder only)
    classified_rasters_dir: Union[str, Path],
    # Processing / outputs
    nodata_out: int = 255,
    overwrite_rasters: bool = False,
    overwrite_models: bool = False,
    # Warnings / checks
    warn_on_unused_bands: bool = True,
) -> pd.DataFrame:
    """
    Train one RandomForest classifier per timeframe on a single training year, apply each
    timeframe-specific model to the same timeframe across all requested years, write classified
    rasters (with class-map tags), and compute a quick validation summary table.

    This function relies on helpers being available in scope:
    - `sentinel_path_builder(base_dir, year, quarter_or_month, filename)`
    - `expected_crs_from_input(stacked_raster_crs)`
    - `import_gpkg_func(path)`
    - `RF_models()`
    - `sample_training_bands_for_RF(...)`
    - `apply_RF_model_to_rstrs(...)`
    - `ml_classification_check(...)`

    Output structure (UPDATED)
    --------------------------
    `classified_rasters_dir` is treated as a parent output directory. Outputs are written under:

        classified_rasters_dir /
            stack_description /
                hab_selection /
                    YEAR_TIMEFRAME /
                        RF_out__STACK__HAB_YEAR_TIMEFRAME_ATTEMPT__CRS__rstr.tif
                    model /
                        <train_split_attempt> /
                            RF_model__STACK__HAB__trainYYYY__TF__ATTEMPT__MODEL__CRS.joblib
                    performance_dfs /
                        RF_validation__STACK__HAB__trainYYYY__MODEL__CRS.csv

    Notes:
    - One model is trained per timeframe (or loaded and reused if it exists and
      `overwrite_models=False`).
    - Validation is filtered by year only (using `val_gdf["years"]`), not by timeframe.

    Parameters
    ----------
    train_split_attempt : str
        Required identifier for the training/validation split attempt (e.g., "at1").
        Models are stored under `model/<train_split_attempt>/`.
        If empty/None, a UserWarning is raised and outputs are placed under `model/_missing_attempt/`.

    Returns
    -------
    pandas.DataFrame
        Validation summary results with one row per (clas_year, timeframe). Columns:
        - train_year
        - clas_year
        - timeframe
        - n
        - acc
        - macro_f1
        - weighted_f1
    """
    stacked_raster_dir = Path(stacked_raster_dir)
    classified_rasters_dir = Path(classified_rasters_dir)
    classified_rasters_dir.mkdir(parents=True, exist_ok=True)

    out_root = classified_rasters_dir / str(stack_description) / str(hab_selection)
    out_root.mkdir(parents=True, exist_ok=True)

    model_dir = out_root / "model"
    model_dir.mkdir(parents=True, exist_ok=True)

    performance_dfs_dir = out_root / "performance_dfs"
    performance_dfs_dir.mkdir(parents=True, exist_ok=True)

    if train_split_attempt is None or str(train_split_attempt).strip() == "":
        warnings.warn(
            "train_split_attempt is empty/None. Models will be stored in model/_missing_attempt/. "
            "Provide train_split_attempt to avoid mixing attempts.",
            category=UserWarning,
        )
        attempt_dirname = "_missing_attempt"
    else:
        attempt_dirname = str(train_split_attempt).strip()

    attempt_model_dir = model_dir / attempt_dirname
    attempt_model_dir.mkdir(parents=True, exist_ok=True)

    # --- import gpkg ---
    train_gdf = import_gpkg_func(train_gpkg_path)
    val_all = import_gpkg_func(val_gpkg_path)

    # --- CRS check (train_gdf vs expected) ---
    exp_crs, exp_epsg, exp_name = expected_crs_from_input(stacked_raster_crs)
    if train_gdf.crs is None:
        raise ValueError("train_gdf.crs is None")
    if train_gdf.crs != exp_crs:
        raise ValueError(f"CRS mismatch: expected {exp_epsg} ({exp_name}), got {train_gdf.crs}")

    # --- optional warning: unused bands ---
    if warn_on_unused_bands:
        used = set(feature_band_names)
        if valid_band_available and valid_band_name is not None:
            used.add(valid_band_name)
        unused = [b for b in all_raster_band_names if b not in used]
        if unused:
            warnings.warn(
                f"Some raster bands are not used. Unused: {unused}. Used: {sorted(used)}",
                category=UserWarning,
            )

    # --- mask handling ---
    valid_mask_band = valid_band_name if valid_band_available else None

    # --- models ---
    models_dict = RF_models()
    if rf_model_selection not in models_dict:
        raise ValueError(f"Unknown rf_model_selection='{rf_model_selection}'. Options: {list(models_dict)}")
    base_model = models_dict[rf_model_selection]

    # --- validation setup ---
    if "years" not in val_all.columns:
        raise ValueError("Validation GDF must have a column 'years' for year filtering.")
    val_all = val_all.copy()
    val_all["years_str"] = val_all["years"].astype(str).str.strip()

    years_int = [int(y) for y in years]
    results = []

    import joblib

    # helper to always add a summary row (including skipped cases)
    def _append_summary_row(*, train_year_, clas_year_, timeframe_, n=None, acc=None, macro_f1=None, weighted_f1=None):
        results.append(
            {
                "train_year": str(train_year_),
                "clas_year": str(clas_year_),
                "timeframe": str(timeframe_).upper(),
                "n": n,
                "acc": acc,
                "macro_f1": macro_f1,
                "weighted_f1": weighted_f1,
            }
        )

    for tf in timeframes:
        tf_u = str(tf).upper()

        # --- train raster for this timeframe ---
        train_raster_path = sentinel_path_builder(stacked_raster_dir, int(train_year), tf, stacked_raster_filename)
        if not train_raster_path.exists():
            print(f"Skip timeframe {tf_u} (missing training raster): {train_raster_path}")
            for y in years_int:
                _append_summary_row(train_year_=train_year, clas_year_=y, timeframe_=tf_u)
            continue

        model_name = (
            f"RF_model__{stack_description}__{hab_selection}__train{int(train_year)}__{tf_u}__"
            f"{attempt_dirname}__{rf_model_selection}__{stacked_raster_crs}.joblib"
        )
        model_path = attempt_model_dir / model_name

        # --- Decide: load existing model vs train new ---
        if model_path.exists() and not overwrite_models:
            loaded_artifact = joblib.load(model_path)
            if not isinstance(loaded_artifact, dict) or "model" not in loaded_artifact:
                raise ValueError(f"Model artifact at {model_path} is not the expected dict with key 'model'.")
            rf_model = loaded_artifact["model"]

            artifact_feature_bands = loaded_artifact.get("feature_bands", None)
            if artifact_feature_bands is not None and list(artifact_feature_bands) != list(feature_band_names):
                raise ValueError(
                    f"Feature band mismatch for loaded model {model_path}.\n"
                    f"Loaded: {artifact_feature_bands}\n"
                    f"Requested: {list(feature_band_names)}"
                )

            valid_mask_band_to_use = loaded_artifact.get("valid_mask_band", valid_mask_band)
            valid_value_to_use = loaded_artifact.get("valid_value", valid_value)

            class_map_dict = loaded_artifact.get("class_map", None)
            if class_map_dict is None:
                le_classes = loaded_artifact.get("le_classes", None)
                if le_classes is None:
                    raise ValueError(
                        f"Loaded model artifact {model_path} has no 'class_map' or 'le_classes'. "
                        "Cannot tag rasters / decode validation predictions reliably."
                    )
                class_map_dict = {str(i): cls for i, cls in enumerate(le_classes)}

            n_classes = len(class_map_dict)
            if n_classes >= nodata_out:
                raise ValueError(
                    f"Loaded model has too many classes ({n_classes}) for nodata_out={nodata_out}. "
                    f"Need n_classes <= {nodata_out-1}."
                )

            print(f"Loaded existing model for timeframe {tf_u}: {model_path}")

        else:
            rf_model = clone(base_model)

            X_train, y_train, train_df, class_map_df, le = sample_training_bands_for_RF(
                gdf=train_gdf,
                raster_path=train_raster_path,
                label_col=gpkg_label_col,
                band_names=list(all_raster_band_names),
                valid_mask_band=valid_mask_band,
                valid_value=valid_value,
                feature_bands=list(feature_band_names),
                le=None,
                assume_polygons=True,
            )

            n_classes = len(le.classes_)
            if n_classes >= nodata_out:
                raise ValueError(
                    f"Too many classes ({n_classes}) for nodata_out={nodata_out}. "
                    f"Need n_classes <= {nodata_out-1} (codes 0..{nodata_out-1})."
                )

            rf_model.fit(X_train, y_train)
            class_map_dict = {str(i): cls for i, cls in enumerate(le.classes_)}

            joblib.dump(
                {
                    "model": rf_model,
                    "le_classes": le.classes_.tolist(),
                    "class_map": class_map_dict,
                    "feature_bands": list(feature_band_names),
                    "all_band_names": list(all_raster_band_names),
                    "valid_mask_band": valid_mask_band,
                    "valid_value": valid_value,
                    "train_year": int(train_year),
                    "timeframe": tf,
                    "crs_code": stacked_raster_crs,
                    "stack_description": stack_description,
                    "hab_selection": hab_selection,
                    "train_split_attempt": train_split_attempt,
                },
                model_path,
            )
            print(f"Saved model: {model_path}")

            valid_mask_band_to_use = valid_mask_band
            valid_value_to_use = valid_value

        # --- apply to all years (same timeframe) + validate ---
        for y in years_int:
            in_stack_path = sentinel_path_builder(stacked_raster_dir, y, tf, stacked_raster_filename)
            if not in_stack_path.exists():
                print(f"Skip (missing): {in_stack_path}")
                _append_summary_row(train_year_=train_year, clas_year_=y, timeframe_=tf_u)
                continue

            out_dir = out_root / f"{int(y)}_{tf_u}"
            out_dir.mkdir(parents=True, exist_ok=True)

            out_name = (
                f"RF_out__{stack_description}__{hab_selection}_{int(y)}_{tf_u}_{train_split_attempt}"
                f"__{stacked_raster_crs}__rstr.tif"
            )
            out_path = out_dir / out_name

            if out_path.exists() and not overwrite_rasters:
                print(f"Skip (exists): {out_path}")
            else:
                print(f"Classifying: {in_stack_path} -> {out_path}")
                apply_RF_model_to_rstrs(
                    rf_model=rf_model,
                    in_stack_path=Path(in_stack_path),
                    out_class_path=Path(out_path),
                    expected_band_names=list(all_raster_band_names),
                    valid_mask_band=valid_mask_band_to_use,
                    valid_value=valid_value_to_use,
                    nodata=nodata_out,
                    feature_bands=tuple(feature_band_names),
                )

                with rasterio.open(out_path, "r+") as dst:
                    dst.update_tags(
                        CLASS_MAP=json.dumps(class_map_dict),
                        NODATA_VALUE=str(dst.nodata),
                        TRAIN_YEAR=str(int(train_year)),
                        TIMEFRAME=tf_u,
                        MODEL=rf_model_selection,
                        HAB_DIVISION=hab_selection,
                        CRS_CODE=stacked_raster_crs,
                        TRAIN_SPLIT_ATTEMPT=str(train_split_attempt),
                    )

            # --- quick validation df row for (y, tf) ---
            val_gdf = val_all[val_all["years_str"] == str(y)]
            if val_gdf.empty:
                print(f"Skip (no validation rows): year={y}, timeframe={tf_u}")
                _append_summary_row(train_year_=train_year, clas_year_=y, timeframe_=tf_u)
                continue

            y_true, y_pred, df_cmp = ml_classification_check(
                gdf=val_gdf,
                raster_path=out_path,
                label_col=gpkg_label_col,
                assume_points=False,
            )
            if len(y_true) == 0:
                print(f"Skip (no valid samples after sampling): year={y}, timeframe={tf_u}")
                _append_summary_row(train_year_=train_year, clas_year_=y, timeframe_=tf_u)
                continue

            _append_summary_row(
                train_year_=train_year,
                clas_year_=y,
                timeframe_=tf_u,
                n=int(len(y_true)),
                acc=float(accuracy_score(y_true, y_pred)),
                macro_f1=float(f1_score(y_true, y_pred, average="macro")),
                weighted_f1=float(f1_score(y_true, y_pred, average="weighted")),
            )

    summary_df = pd.DataFrame(
        results,
        columns=["train_year", "clas_year", "timeframe", "n", "acc", "macro_f1", "weighted_f1"],
    )

    summary_csv_name = (
        f"RF_validation__{stack_description}__{hab_selection}__train{int(train_year)}__"
        f"{rf_model_selection}__{stacked_raster_crs}.csv"
    )
    summary_df.to_csv(performance_dfs_dir / summary_csv_name, index=False)

    return summary_df

In [1]:
from functions.gpkg_funcs import (
    import_gpkg_func,
)

from functions.raster_io_funcs import (
    print_raster_info,
    import_raster_func,
)

from functions.s2_import_funcs import (
    sentinel_path_builder,
)

from functions.ML_classification_check_funcs import (
    ml_classification_check,
)

from notebooks_dir._04_ML_classification._support._n01_funcs import (
    expected_crs_from_input,
    RF_models,
    sample_training_bands_for_RF,
    apply_RF_model_to_rstrs,
)

## Setting directory

In [2]:
from paths.OG_paths import (
    #######################################
    #  03_training_validation_data_split  #
    #######################################
    # |   n03_training_validation_split   |
    # +===================================+
    # --- training validation GPKGs UTM32631 --- #
    # WD1
    training_pixels__WD1_plusOW_p80_tmp2_ML_at1__gelderland__UTM32631__gpkg_path,
    validation_pixels__WD1_plusOW_p80_tmp2_ML_at1__gelderland__UTM32631__gpkg_path,
    # WD2
    training_pixels__WD2_plusOW_p80_tmp2_ML_at1__gelderland__UTM32631__gpkg_path,
    validation_pixels__WD2_plusOW_p80_tmp2_ML_at1__gelderland__UTM32631__gpkg_path,
    # WD3
    training_pixels__WD3_plusOW_p80_tmp2_ML_at1__gelderland__UTM32631__gpkg_path,
    validation_pixels__WD3_plusOW_p80_tmp2_ML_at1__gelderland__UTM32631__gpkg_path,
    # WD4
    training_pixels__WD4_plusOW_p80_tmp2_ML_at1__gelderland__UTM32631__gpkg_path,
    validation_pixels__WD4_plusOW_p80_tmp2_ML_at1__gelderland__UTM32631__gpkg_path,
    # WD5
    training_pixels__WD5_plusOW_p80_tmp2_ML_at1__gelderland__UTM32631__gpkg_path,
    validation_pixels__WD5_plusOW_p80_tmp2_ML_at1__gelderland__UTM32631__gpkg_path,

    #######################################
    #        04_ML_classification         #
    #######################################ac
    # |  n01_random_forest_s2_quarterly   |
    # +===================================+
    # DIRs
    s2_stack_b2348__dir,
    s2_stack_b2ndvwi__dir,

    # --- s2 2017 quarterly input --- #
    s2_stack_b2348__veluwe__2017_q1__UTM32631__rstr_path,
    s2_stack_b2348__veluwe__2017_q2__UTM32631__rstr_path,
    s2_stack_b2348__veluwe__2017_q3__UTM32631__rstr_path,
    s2_stack_b2348__veluwe__2017_q4__UTM32631__rstr_path,

    # --- RF model output --- #
    RF_out__b2348__WD1__UTM32631__rstrs_dir,
)

## Importing packages

In [3]:
import numpy as np
import pandas as pd
import rasterio
import json
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
le = LabelEncoder()

## RF classifyers

_here will an explanation be on the different classifyers. What are they taylored to?_ <br>

Currently available models: <br>
- rf_model_1 -> "mdl1"

## RF Q1 training

_attempt 1 -> rf-model1 for b2b3b4b8_

In [4]:
# --- Input vars --- #
# training raster stack
stacked_raster_dir = s2_stack_b2348__dir
stacked_raster_crs = "UTM32631"     # OR "RD" OR "WGS84"
stacked_raster_filename = "S2_b2348_stack.tif"
all_raster_band_names = ["B02", "B03", "B04", "B08", "VALID_MASK"]
feature_band_names = ["B02", "B03", "B04", "B08"]
stacked_rsrs_years = ['2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
stacked_rsrs_timeframe = ['q1', 'q2', 'q3', 'q4']   # OR ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12']

valid_band_available = True
valid_band_name = "VALID_MASK"
valid_value = 1

# RF-training
rf_model_selection = "mdl1"
train_year = 2017
hab_selection = "WD1"

# train/validation gpkg paths
train_gpkg_path = training_pixels__WD1_plusOW_p80_tmp2_ML_at1__gelderland__UTM32631__gpkg_path
val_gpkg_path = validation_pixels__WD1_plusOW_p80_tmp2_ML_at1__gelderland__UTM32631__gpkg_path
gpkg_label_col = "type"

# classified rasters
classified_rasters_dir = RF_out__b2348__WD1__UTM32631__rstrs_dir

In [ ]:
# --- Setting directory --- #
# stacked input data (single year)
train_raster_path = sentinel_path_builder(stacked_raster_dir, train_year, stacked_rsrs_timeframe, stacked_raster_filename)

# saved RF-model
RF_model_out__dir = classified_rasters_dir / "model"
RF_model_out__dir.mkdir(parents=True, exist_ok=True)
RF_training_results__dir = RF_model_out__dir / "performance_df"
RF_training_results__dir.mkdir(parents=True, exist_ok=True)

# saved classified rasters
classified_rasters_dir.mkdir(parents=True, exist_ok=True)

In [6]:
# --- importing files --- #
train_gdf = import_gpkg_func(train_gpkg_path);
val_gdf = import_gpkg_func(val_gpkg_path);

# Check if the crs matches the expected crs
exp_crs, exp_epsg, exp_name = expected_crs_from_input(stacked_raster_crs)

if train_gdf.crs is None:
    raise ValueError("train_gdf.crs is None")
if train_gdf.crs != exp_crs:
    raise ValueError(f"CRS mismatch: expected {exp_epsg} ({exp_name}), got {train_gdf.crs}")

c:\Users\NL1G3K\Desktop\Vegetation_quality_monitoring\.pixi\envs\default\Lib\site-packages\pyogrio\core.py:23: RuntimeWarning: Could not detect GDAL data files.  Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()
c:\Users\NL1G3K\Desktop\Vegetation_quality_monitoring\.pixi\envs\default\Lib\site-packages\pyogrio\core.py:24: RuntimeWarning: Could not detect PROJ data files.  Set PROJ_LIB environment variable to the correct path.
  _init_proj_data()


In [7]:
# --- RF-training --- #
# Get the selected model
models_dict = RF_models()
rf_model = models_dict[rf_model_selection]

# Build a training array for RF
X_train, y_train, train_df, class_map_df, le = sample_training_bands_for_RF(
    gdf=train_gdf,
    raster_path=train_raster_path,
    label_col=gpkg_label_col,
    band_names=all_raster_band_names,
    valid_mask_band=valid_band_name,
    valid_value=valid_value,
    feature_bands=feature_band_names,
)

In [10]:
X_train

array([[ 258,  455,  492, 1880],
       [ 222,  377,  353, 1946],
       [ 156,  254,  260, 1546],
       ...,
       [ 243,  386,  364,  343],
       [ 489,  621,  616,  559],
       [ 125,  132,  197,  195]], dtype=int16)

In [11]:
y_train

array([1, 1, 1, ..., 0, 0, 0])

In [8]:
# Training the RF model
rf_model.fit(X_train, y_train);

In [9]:
# Make a classes dict to add to rstr metadata
class_map_dict = {str(i): cls for i, cls in enumerate(le.classes_)}

In [ ]:
# Apply the trained RF model on the rasters of all years
for y in stacked_rsrs_years:
    for p in stacked_rsrs_timeframe:
        # input rasters
        stacked_rstrs_path = sentinel_path_builder(stacked_raster_dir, int(y), stacked_rsrs_timeframe(p), stacked_raster_filename)
        # output rasters
        classified_rstr_name = f"RF_out__b2348__{hab_selection}_{int(y)}_{stacked_rsrs_timeframe(p)}_{rf_model_selection}__{stacked_raster_crs}__rstr.tif"
        classified_rstr_path = sentinel_path_builder(classified_rasters_dir, int(y), stacked_rsrs_timeframe(p), classified_rstr_name)

        if not Path(stacked_rstrs_path).exists():
            print(f"Skip (missing): {stacked_rstrs_path}")
            continue

        print(f"Classifying: {stacked_rstrs_path} -> {classified_rstr_path}")
        apply_RF_model_to_rstrs(
            rf_model=rf_model,
            in_stack_path=Path(stacked_rstrs_path),
            out_class_path=classified_rstr_path,
            valid_mask_band=valid_band_name,
            valid_value=valid_value,
            nodata=255,
            feature_bands=("B02","B03","B04","B08"),
        )

        # add the classes to raster tag
        with rasterio.open(classified_rstr_path, "r+") as dst:
            dst.update_tags(
                CLASS_MAP=json.dumps(class_map_dict),
                LE_CLASSES=json.dumps(le.classes_.tolist()),
                NODATA_VALUE=str(dst.nodata),
            )

Classifying: D:\Thesis\10.Thesis_Data\01_remote_sensed_processed\s2_mosaic\04_b2b3b4b8_stack\2017_Q1\S2_b2348_stack.tif -> D:\Thesis\10.Thesis_Data\04_RF_output\WD1\2017_Q1\RF_out__b2348__WD1_2017_Q1_mdl1__UTM32631__rstr.tif


NameError: name 'apply_RF_model_to_rstrs' is not defined

In [12]:
val_gdf.head()

,type,years,geometry
index,,,
GI_19657_1,Remaining,2018,"POLYGON ((688620 5787350, 688610 5787350, 6886..."
GI_19657_2,Remaining,2018,"POLYGON ((688620 5787360, 688610 5787360, 6886..."
GI_19657_3,Remaining,2018,"POLYGON ((688630 5787360, 688620 5787360, 6886..."
GI_19697_1,Remaining,2018,"POLYGON ((688940 5787630, 688930 5787630, 6889..."
GI_19697_2,Remaining,2018,"POLYGON ((688940 5787640, 688930 5787640, 6889..."


In [ ]:
# --- build "quick validation df" --- #
results = []

val_all = validation_pixels__WD1_plusOW_p80_tmp2_ML_at1__gelderland__UTM32631__gdf.copy()
val_all["years_str"] = val_all["years"].astype(str).str.strip()

for y in s2_years_list:
    y_str = str(y).strip()

    raster_path = Path(classified_rasters_dir) / f"{y}_{q}" / f"RF_out__b2348__WD1_{y}_{q}_at1__UTM32631__rstr.tif"
    if not raster_path.exists():
        print("Missing:", raster_path)
        continue

    val_gdf = val_all[val_all["years_str"] == y_str]
    if val_gdf.empty:
        print(f"Skip (no validation rows): year={y_str}, quarter={q}")
        continue

    y_true, y_pred, df_cmp = ml_classification_check(
        gdf=val_gdf,
        raster_path=raster_path,
        label_col="type",
        assume_points=False,
    )
    if len(y_true) == 0:
        print(f"Skip (no valid samples after sampling): year={y_str}, quarter={q}")
        continue

    results.append({
        "year": int(y),
        "quarter": q,
        "n": len(y_true),
        "acc": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    })

results_df = pd.DataFrame(results)
results_df

Skip (no validation rows): year=2017, quarter=Q1


,year,quarter,n,acc,macro_f1,weighted_f1
0,2018,Q1,501,0.524950,0.301612,0.437588
1,2019,Q1,274,0.952555,0.650434,0.973865
2,2020,Q1,260,0.500000,0.493423,0.341242
3,2021,Q1,157,0.980892,0.495177,0.990354
4,2022,Q1,52,0.480769,0.324675,0.312188
5,2023,Q1,186,0.973118,0.661017,0.986331
6,2024,Q1,31,1.000000,1.000000,1.000000


## RF Q2 training

In [ ]:
# --- Input vars --- #
# training raster stack
stacked_raster_dir = s2_stack_b2348__dir
stacked_raster_filename = "S2_b2348_stack.tif"
raster_band_names = ["B02", "B03", "B04", "B08", "VALID_MASK"]

# RF-training
rf_model = rf_model1
train_year = 2017
train_val_q = "Q1"

# train/validation gdfs
train_gdf = training_pixels__WD1_plusOW_p80_tmp2_ML_at1__gelderland__UTM32631__gdf
val_gdf = validation_pixels__WD1_plusOW_p80_tmp2_ML_at1__gelderland__UTM32631__gdf
label_col = "type"

# classified rasters
classified_rasters_dir = Path(r"D:\Thesis\10.Thesis_Data\04_RF_output\WD1")